In [1]:
%pip install pandas numpy scikit-learn catboost lightgbm nbformat

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, CatBoostRegressor
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, LabelEncoder
from sklearn.metrics import mean_absolute_error
import plotly.express as px

In [3]:
train_df = pd.read_csv('./datasets/train.csv', sep=';', decimal=',', na_values=['None', 'nan'], low_memory=False)
test_df = pd.read_csv('./datasets/test.csv', sep=';', decimal=',', na_values=['None', 'nan'], low_memory=False)

In [4]:
def coerce_dtypes(df):
    df = df.copy()
    for col in df.select_dtypes(include=['str']).columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='raise').astype('float64')
        except:
            df[col] = df[col].astype('str')
    return df

dtype_coercer = FunctionTransformer(coerce_dtypes)

train_df = dtype_coercer.fit_transform(train_df)
test_df = dtype_coercer.transform(test_df)

train_df['dt'] = pd.to_datetime(train_df['dt'], format='%Y-%m-%d')

In [5]:
drop_cols = [
    'id', 'dt',
    'target', 'w',
    'dp_address_unique_regions',
    'first_salary_income',
    'period_last_act_ad'
]

X = train_df.drop(drop_cols, axis=1).copy()
y = train_df['target']
w = train_df['w']

cat_cols = X.select_dtypes(include=['str']).columns.tolist()

# Preprocessing

Общая импутация через CatBoost (он нативно работает со строками).
Затем две ветки:
- **CatBoost** — получает данные со строковыми колонками (как есть)
- **LightGBM** — получает данные после LabelEncoding

In [6]:
class CatFillna(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols):
        self.cat_cols = cat_cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cat_cols:
            X[col] = X[col].fillna('missing').astype(str)
        return X

In [7]:
class OOFTargetImputer(BaseEstimator, TransformerMixin):
    def __init__(self, target_col, feature_cols, cat_features, n_splits=5, seed=42):
        self.target_col = target_col
        self.feature_cols = feature_cols
        self.cat_features = cat_features
        self.n_splits = n_splits
        self.seed = seed

    def fit(self, X, y=None):
        mask_known = X[self.target_col].notna()
        X_known = X.loc[mask_known, self.feature_cols]
        y_known = X.loc[mask_known, self.target_col]
        self.model_ = CatBoostRegressor(
            iterations=1000, depth=6, learning_rate=0.05,
            loss_function='RMSE', random_seed=self.seed, verbose=False
        )
        self.model_.fit(X_known, y_known, cat_features=self.cat_features)
        return self

    def transform(self, X):
        X = X.copy()
        mask_missing = X[self.target_col].isna()
        X[f'{self.target_col}_missing'] = mask_missing.astype(int)
        if mask_missing.any():
            X.loc[mask_missing, self.target_col] = self.model_.predict(
                X.loc[mask_missing, self.feature_cols]
            )
        return X

    def fit_transform(self, X, y=None):
        self.fit(X, y)
        X = X.copy()
        mask_known = X[self.target_col].notna()
        mask_missing = ~mask_known
        X_known = X.loc[mask_known, self.feature_cols]
        y_known = X.loc[mask_known, self.target_col]
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.seed)
        oof_pred = np.zeros(len(X_known))
        for tr_idx, ho_idx in kf.split(X_known):
            m = CatBoostRegressor(
                iterations=1000, depth=6, learning_rate=0.05,
                loss_function='RMSE', random_seed=self.seed, verbose=False
            )
            m.fit(X_known.iloc[tr_idx], y_known.iloc[tr_idx], cat_features=self.cat_features)
            oof_pred[ho_idx] = m.predict(X_known.iloc[ho_idx])
        mae_imputer = mean_absolute_error(y_known, oof_pred)
        print(f'Imputer MAE (OOF): {mae_imputer:.0f}  (mean target: {y_known.mean():.0f})')
        imputed_values = pd.Series(index=X.index, dtype=float)
        imputed_values.loc[mask_known] = oof_pred
        imputed_values.loc[mask_missing] = self.model_.predict(X.loc[mask_missing, self.feature_cols])
        X[f'{self.target_col}_missing'] = mask_missing.astype(int)
        X[self.target_col] = imputed_values
        return X

In [ ]:
class CatLabelEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.encoders_ = {}
        for col in X.select_dtypes(include=['str']).columns:
            filled = X[col].fillna('__nan__').astype(str)
            le = LabelEncoder()
            le.fit(filled)
            self.encoders_[col] = le
        return self

    def transform(self, X):
        X = X.copy()
        for col, le in self.encoders_.items():
            if col not in X.columns:
                continue
            filled = X[col].fillna('__nan__').astype(str)
            known = set(le.classes_)
            filled = filled.apply(lambda v: v if v in known else '__unknown__')
            le_full = LabelEncoder()
            le_full.classes_ = np.append(le.classes_, '__unknown__')
            X[col] = le_full.transform(filled)
        return X

In [ ]:
imputer_features = [c for c in X.columns if c not in [
    'salary_6to12m_avg', 'incomeValue', 'dp_ils_avg_salary_1y',
    'dp_ils_avg_salary_2y', 'dp_ils_avg_salary_3y',
    'dp_payoutincomedata_payout_avg_3_month', 'dp_payoutincomedata_payout_avg_6_month',
    'dp_payoutincomedata_payout_avg_prev_year', 'first_salary_income'
]]

cat_cols_for_imputer = [c for c in cat_cols if c in imputer_features]

In [ ]:
impute_pipeline = Pipeline([
    ('cat_fillna', CatFillna(cat_cols=cat_cols)),
    ('salary_impute', OOFTargetImputer(
        target_col='salary_6to12m_avg',
        feature_cols=imputer_features,
        cat_features=cat_cols_for_imputer,
    )),
])

X_imputed = impute_pipeline.fit_transform(X)

# Ветка LightGBM: label-encode оставшиеся строки
lgb_pipeline = Pipeline([
    ('cat_label_encode', CatLabelEncoder()),
])
X_lgb = lgb_pipeline.fit_transform(X_imputed)

# Ветка CatBoost: оставляем строки как есть
X_cb = X_imputed

Imputer MAE (OOF): 32525  (mean target: 117490)


# Train / Val split

In [11]:
val_cutoff = pd.Timestamp('2024-06-01')
train_mask_time = (train_df['dt'] < val_cutoff).values
val_mask_time = (train_df['dt'] >= val_cutoff).values

In [12]:
def split(X_full, mask):
    return X_full[mask].reset_index(drop=True)

y_train = y[train_mask_time].reset_index(drop=True)
y_val = y[val_mask_time].reset_index(drop=True)
w_train = w[train_mask_time].reset_index(drop=True)
w_val = w[val_mask_time].reset_index(drop=True)

# CatBoost ветка
cb_X_train = split(X_cb, train_mask_time)
cb_X_val = split(X_cb, val_mask_time)

# LightGBM ветка
lgb_X_train = split(X_lgb, train_mask_time)
lgb_X_val = split(X_lgb, val_mask_time)

In [13]:
def make_regime_4(y):
    return pd.cut(y, bins=[0, 83000, 200000, 400000, np.inf], labels=[0,1,2,3]).astype(int)

regime_train_4 = make_regime_4(y_train)
regime_val_4 = make_regime_4(y_val)
print(regime_train_4.value_counts().sort_index())

target
0    40123
1    15710
2     3449
3     1290
Name: count, dtype: int64


# CatBoost models

In [14]:
cb_clf = CatBoostClassifier(
    learning_rate=0.051627416448269105,
    depth=8,
    l2_leaf_reg=7.9142461137166755,
    min_data_in_leaf=120,
    random_strength=3.4495138326286,
    bagging_temperature=0.4948872578136553,
    iterations=4000,
    early_stopping_rounds=50,
    task_type='GPU',
    loss_function='MultiClass',
    eval_metric='MultiClass',
    use_best_model=True,
    verbose=100,
)

cb_clf.fit(
    cb_X_train, regime_train_4,
    sample_weight=w_train,
    cat_features=cat_cols,
    eval_set=(cb_X_val, regime_val_4)
)

0:	learn: 1.3252974	test: 1.3165809	best: 1.3165809 (0)	total: 40.9ms	remaining: 2m 43s
100:	learn: 0.6793183	test: 0.6194477	best: 0.6194456 (99)	total: 1.66s	remaining: 1m 4s
200:	learn: 0.6377998	test: 0.6037721	best: 0.6037721 (200)	total: 2.85s	remaining: 53.8s
300:	learn: 0.5909058	test: 0.5917315	best: 0.5917315 (300)	total: 4.7s	remaining: 57.8s
400:	learn: 0.5549733	test: 0.5847498	best: 0.5847498 (400)	total: 6.6s	remaining: 59.2s
500:	learn: 0.5276940	test: 0.5805741	best: 0.5805741 (500)	total: 8.43s	remaining: 58.9s
600:	learn: 0.5055489	test: 0.5768590	best: 0.5768590 (600)	total: 10.3s	remaining: 58.1s
700:	learn: 0.4878404	test: 0.5741503	best: 0.5741503 (700)	total: 12.1s	remaining: 56.7s
800:	learn: 0.4705030	test: 0.5714602	best: 0.5714602 (800)	total: 13.9s	remaining: 55.4s
900:	learn: 0.4553240	test: 0.5691727	best: 0.5691727 (900)	total: 15.7s	remaining: 53.9s
1000:	learn: 0.4409326	test: 0.5670614	best: 0.5670492 (998)	total: 17.5s	remaining: 52.4s
1100:	learn: 0

CatBoostClassifier(bagging_temperature=0.4948872578136553, depth=8, early_stopping_rounds=50, eval_metric='MultiClass', iterations=4000, l2_leaf_reg=7.9142461137166755, learning_rate=0.051627416448269105, loss_function='MultiClass', min_data_in_leaf=120, random_strength=3.4495138326286, task_type='GPU', use_best_model=True, verbose=100)

In [15]:
cb_regressor_configs = {
    0: dict(depth=6, l2_leaf_reg=3.0, min_data_in_leaf=50, learning_rate=0.03),
    1: dict(depth=6, l2_leaf_reg=4.0, min_data_in_leaf=30, learning_rate=0.03),
    2: dict(depth=5, l2_leaf_reg=6.0, min_data_in_leaf=20, learning_rate=0.04),
    3: dict(depth=4, l2_leaf_reg=10.0, min_data_in_leaf=10, learning_rate=0.05),
}

cb_regressors = {}
for regime_id, cfg in cb_regressor_configs.items():
    mask_tr = regime_train_4 == regime_id
    mask_vl = regime_val_4 == regime_id
    m = CatBoostRegressor(
        **cfg, eval_metric='MAE', iterations=4000,
        early_stopping_rounds=50, verbose=100, task_type='GPU'
    )
    m.fit(
        cb_X_train[mask_tr], y_train[mask_tr],
        sample_weight=w_train[mask_tr],
        cat_features=cat_cols,
        eval_set=(cb_X_val[mask_vl], y_val[mask_vl])
    )
    cb_regressors[regime_id] = m
    print(f'CB regime {regime_id}: n={mask_tr.sum()}')

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 12427.3186052	test: 16663.3025947	best: 16663.3025947 (0)	total: 52.4ms	remaining: 3m 29s
100:	learn: 10706.8798410	test: 13878.3543028	best: 13878.3543028 (100)	total: 3.38s	remaining: 2m 10s
200:	learn: 10454.1975398	test: 13507.1077095	best: 13507.1077095 (200)	total: 6.69s	remaining: 2m 6s
300:	learn: 10315.0555649	test: 13329.3052081	best: 13329.3052081 (300)	total: 10.1s	remaining: 2m 4s
400:	learn: 10213.9767104	test: 13235.3929438	best: 13235.3929438 (400)	total: 13.8s	remaining: 2m 3s
500:	learn: 10122.0334869	test: 13153.8950905	best: 13153.7771141 (498)	total: 17.5s	remaining: 2m 2s
600:	learn: 10047.5938099	test: 13099.0330409	best: 13099.0330409 (600)	total: 21.2s	remaining: 1m 59s
700:	learn: 9980.4279606	test: 13047.9581856	best: 13047.8387157 (698)	total: 24.9s	remaining: 1m 57s
800:	learn: 9917.0793643	test: 13008.1568042	best: 13008.1568042 (800)	total: 28.6s	remaining: 1m 54s
900:	learn: 9860.9109493	test: 12968.9602389	best: 12968.8990106 (899)	total: 32.3

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 25160.4930633	test: 35567.1506722	best: 35567.1506722 (0)	total: 29.8ms	remaining: 1m 59s
100:	learn: 22120.5051863	test: 29803.7125637	best: 29803.7125637 (100)	total: 3.18s	remaining: 2m 2s
200:	learn: 21507.1991827	test: 28860.3356514	best: 28860.3356514 (200)	total: 6.21s	remaining: 1m 57s
300:	learn: 21042.8636422	test: 28380.1427909	best: 28380.1427909 (300)	total: 9.26s	remaining: 1m 53s
400:	learn: 20682.5140079	test: 28117.9527121	best: 28117.9527121 (400)	total: 12.5s	remaining: 1m 51s
500:	learn: 20339.9898000	test: 27897.3834029	best: 27897.3834029 (500)	total: 15.6s	remaining: 1m 49s
600:	learn: 20045.9711636	test: 27761.7654149	best: 27761.7654149 (600)	total: 18.7s	remaining: 1m 45s
700:	learn: 19764.1879369	test: 27612.0686138	best: 27612.0686138 (700)	total: 21.7s	remaining: 1m 42s
800:	learn: 19515.6525516	test: 27504.9012517	best: 27504.9012517 (800)	total: 24.7s	remaining: 1m 38s
900:	learn: 19276.0843880	test: 27420.5248030	best: 27420.5248030 (900)	total

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 46610.7411797	test: 48799.7813953	best: 48799.7813953 (0)	total: 20.9ms	remaining: 1m 23s
100:	learn: 40414.5481265	test: 43299.0697674	best: 43299.0697674 (100)	total: 1.97s	remaining: 1m 16s
200:	learn: 38641.5697558	test: 42747.0418605	best: 42742.0325581 (199)	total: 3.87s	remaining: 1m 13s
300:	learn: 37209.4654453	test: 42638.9488372	best: 42636.8372093 (291)	total: 5.73s	remaining: 1m 10s
400:	learn: 36237.8580361	test: 42678.3860465	best: 42624.0837209 (352)	total: 7.68s	remaining: 1m 8s
bestTest = 42624.08372
bestIteration = 352
Shrink model to first 353 iterations.
CB regime 2: n=3449


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 213636.8320181	test: 216426.3558282	best: 216426.3558282 (0)	total: 19.5ms	remaining: 1m 18s
100:	learn: 156636.8751496	test: 190658.5030675	best: 190658.5030675 (100)	total: 1.58s	remaining: 1m 1s
200:	learn: 143184.2500775	test: 187210.9693252	best: 187208.1840491 (199)	total: 3.17s	remaining: 59.8s
300:	learn: 133642.5903770	test: 186125.9386503	best: 185988.8588957 (279)	total: 5s	remaining: 1m 1s
400:	learn: 124386.5968609	test: 184710.3067485	best: 184618.4539877 (388)	total: 6.82s	remaining: 1m 1s
bestTest = 184618.454
bestIteration = 388
Shrink model to first 389 iterations.
CB regime 3: n=1290


# LightGBM models

In [16]:
lgb_clf = lgb.LGBMClassifier(
    n_estimators=4000,
    objective='multiclass',
    random_state=42,
    verbose=-1,
)

lgb_clf.fit(
    lgb_X_train, regime_train_4,
    sample_weight=w_train,
    eval_set=(lgb_X_val, regime_val_4),
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

Training until validation scores don't improve for 50 rounds
[100]	valid_0's multi_logloss: 0.564942
[200]	valid_0's multi_logloss: 0.556503
[300]	valid_0's multi_logloss: 0.555046
Early stopping, best iteration is:
[274]	valid_0's multi_logloss: 0.554207


,n_estimators,4000
,objective,'multiclass'
,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,subsample_for_bin,200000
,class_weight,None
,min_split_gain,0.0


In [17]:
lgb_regressor_configs = {
    0: dict(max_depth=6, num_leaves=63, reg_lambda=3.0, min_child_samples=50, learning_rate=0.03),
    1: dict(max_depth=6, num_leaves=63, reg_lambda=4.0, min_child_samples=30, learning_rate=0.03),
    2: dict(max_depth=5, num_leaves=31, reg_lambda=6.0, min_child_samples=20, learning_rate=0.04),
    3: dict(max_depth=4, num_leaves=15, reg_lambda=10.0, min_child_samples=10, learning_rate=0.05),
}

lgb_regressors = {}
for regime_id, cfg in lgb_regressor_configs.items():
    mask_tr = regime_train_4 == regime_id
    mask_vl = regime_val_4 == regime_id
    m = lgb.LGBMRegressor(
        **cfg, metric='mae', n_estimators=4000,
        verbose=-1, random_state=42
    )
    m.fit(
        lgb_X_train[mask_tr], y_train[mask_tr],
        sample_weight=w_train[mask_tr],
        eval_set=(lgb_X_val[mask_vl], y_val[mask_vl]),
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )
    lgb_regressors[regime_id] = m
    print(f'LGB regime {regime_id}: n={mask_tr.sum()}')

Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 13278.9
[200]	valid_0's l1: 12984.6
[300]	valid_0's l1: 12868.8
[400]	valid_0's l1: 12815
[500]	valid_0's l1: 12761.7
[600]	valid_0's l1: 12720.3
[700]	valid_0's l1: 12676.1
[800]	valid_0's l1: 12635.3
[900]	valid_0's l1: 12605.1
[1000]	valid_0's l1: 12577.3
[1100]	valid_0's l1: 12552.9
[1200]	valid_0's l1: 12532.7
[1300]	valid_0's l1: 12514.7
[1400]	valid_0's l1: 12492.6
[1500]	valid_0's l1: 12473.9
[1600]	valid_0's l1: 12459.7
[1700]	valid_0's l1: 12444
[1800]	valid_0's l1: 12428.9
[1900]	valid_0's l1: 12409.7
[2000]	valid_0's l1: 12397.5
[2100]	valid_0's l1: 12379.6
[2200]	valid_0's l1: 12365.8
[2300]	valid_0's l1: 12351.3
[2400]	valid_0's l1: 12341.4
[2500]	valid_0's l1: 12329.4
[2600]	valid_0's l1: 12320.8
[2700]	valid_0's l1: 12312.6
[2800]	valid_0's l1: 12302.1
[2900]	valid_0's l1: 12291.3
[3000]	valid_0's l1: 12282.3
[3100]	valid_0's l1: 12271.6
[3200]	valid_0's l1: 12265.8
[3300]	valid_0's l1: 122

# Ensemble: predict + blend

In [18]:
def predict_mixture(X, classifier, regressors_dict, regime_ids):
    proba = classifier.predict_proba(X)
    preds = np.column_stack([regressors_dict[i].predict(X) for i in regime_ids])
    return (proba * preds).sum(axis=1)

def wmae(y_true, y_pred, weight):
    return np.average(np.abs(y_true - y_pred), weights=weight)

def weighted_median(values, weights):
    order = np.argsort(values)
    v, wt = np.asarray(values)[order], np.asarray(weights)[order]
    return v[np.searchsorted(np.cumsum(wt), wt.sum() / 2)]

def affine_cal(y_t, p, wt, lo=0.90, hi=1.15, n=51):
    best = (np.inf, 1.0, 0.0)
    for s in np.linspace(lo, hi, n):
        sh = weighted_median(y_t - s * p, wt)
        sc = wmae(y_t, np.clip(s * p + sh, y_t.min(), y_t.max()), wt)
        if sc < best[0]:
            best = (sc, float(s), float(sh))
    return best

In [19]:
# CatBoost predictions
pred_cb = predict_mixture(cb_X_val, cb_clf, cb_regressors, [0,1,2,3])

# LightGBM predictions
pred_lgb = predict_mixture(lgb_X_val, lgb_clf, lgb_regressors, [0,1,2,3])

print(f'CatBoost WMAE:  {wmae(y_val, pred_cb, w_val):.2f}')
print(f'LightGBM WMAE:  {wmae(y_val, pred_lgb, w_val):.2f}')
print(f'Plain average:  {wmae(y_val, (pred_cb + pred_lgb) / 2, w_val):.2f}')

CatBoost WMAE:  62841.26
LightGBM WMAE:  63431.25
Plain average:  62543.41


In [20]:
# Подбор оптимальных весов ансамбля на validation
best_score = np.inf
best_alpha = 0.5
for alpha in np.linspace(0.0, 1.0, 101):
    blended = alpha * pred_cb + (1 - alpha) * pred_lgb
    score = wmae(y_val, blended, w_val)
    if score < best_score:
        best_score = score
        best_alpha = alpha

print(f'Optimal weight: CatBoost={best_alpha:.2f}, LightGBM={1-best_alpha:.2f}')
print(f'Ensemble WMAE:  {best_score:.2f}')

pred_ensemble = best_alpha * pred_cb + (1 - best_alpha) * pred_lgb

Optimal weight: CatBoost=0.63, LightGBM=0.37
Ensemble WMAE:  62505.65


In [ ]:
cal_wmae, CAL_SCALE, CAL_SHIFT = affine_cal(y_val.to_numpy(), pred_ensemble, w_val.to_numpy())
print(f'До калибровки:  {wmae(y_val, pred_ensemble, w_val):.2f}')
print(f'После калибровки: {cal_wmae:.2f} (scale={CAL_SCALE:.4f}, shift={CAL_SHIFT:.0f})')

Do калибровки:  62505.65
После калибровки: 62238.95 (scale=1.0200, shift=-5117)


# Submit

In [23]:
Y_raw = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])

# Общая импутация
Y_imputed = impute_pipeline.transform(Y_raw)

# CatBoost ветка
Y_cb = Y_imputed

# LightGBM ветка
Y_lgb = lgb_pipeline.transform(Y_imputed)

# Предсказания
pred_test_cb = predict_mixture(Y_cb, cb_clf, cb_regressors, [0,1,2,3])
pred_test_lgb = predict_mixture(Y_lgb, lgb_clf, lgb_regressors, [0,1,2,3])

# Ансамбль
pred_test = best_alpha * pred_test_cb + (1 - best_alpha) * pred_test_lgb

# Калибровка
month_h_map = {'2024-07-31': 1, '2024-08-31': 2, '2024-09-30': 3, '2024-10-31': 4, '2024-11-30': 5}
RATE = 0.01
h = test_df['dt'].astype(str).map(month_h_map).to_numpy()
prog_scale = CAL_SCALE + RATE * (h - 1)

pred_test_calibrated = np.clip(
    prog_scale * pred_test + CAL_SHIFT,
    y_train.min(), y_train.max()
)

submit = pd.DataFrame({
    'id': test_df['id'],
    'predict': pred_test_calibrated
}).set_index('id').to_csv('submit_ensemble.csv', decimal=',', sep=';')

print(f'Submit saved. Weight: CB={best_alpha:.2f}, LGB={1-best_alpha:.2f}')

Submit saved. Weight: CB=0.63, LGB=0.37
